# LIMINAL — Phase 9 Experiment Notebook

**Project**: Adaptive Internal–External Latent Workspace (LIMINAL)  
**Phase**: 9 — Hypothesis A & B validation with new task families  

---
**How to use this notebook**

- Run sections **sequentially**. Each section depends on outputs from the previous.
- Skip sections marked `[OPTIONAL]` on first pass.
- Outputs (checkpoints, JSON reports, plots) are saved to `results/` and will persist between runs.
- On **Colab**: mount Drive first (Cell 0), then run from top. On **local**: skip Cell 0.
- All commands are wrapped in `!` (shell) or `%%bash`. Toggle between as needed.

---
**Phases covered**

| Section | What it does | GPU needed? |
|---------|-------------|-------------|
| 0 | Setup & drive mount | No |
| 1 | Phase 8 housekeeping (P0 metric reconciliation) | No |
| 2 | Phase 8 housekeeping (P3 seed-43 reruns) | Yes |
| 3 | Task generator validation (no training) | No |
| 4 | Hypothesis A training — relational_graph, 3 models | Yes |
| 5 | Hypothesis A eval — difficulty ladder + inference scaling | No |
| 6 | Hypothesis A eval — slot specialisation probe | No |
| 7 | Hypothesis B training — obligation_network | Yes |
| 8 | Hypothesis B eval — 4-condition write-policy comparison | No |
| 9 | Hypothesis B eval — demand-sensitivity + per-turn ablation | No |
| 10 | Summary scorecard | No |

---
## Section 0 — Environment Setup

Run **one** of the two cells below depending on your environment.

In [1]:
# ── Colab only: mount Drive and cd to project ─────────────────────────────
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/LIMINAL'
    os.chdir(PROJECT_ROOT)
    print(f'Working directory: {os.getcwd()}')
else:
    # Local: just confirm we're in the right place
    PROJECT_ROOT = os.getcwd()
    assert os.path.exists('src/data/arithmetic.py'), \
        'Run this notebook from the LIMINAL project root!'
    print(f'Working directory: {PROJECT_ROOT}')

In [ ]:
# OPTION A: Clone your repo from GitHub (replace with your repo URL if applicable)
!git clone https://github.com/isaac-hash/LIMINAL.git /content/LIMINAL
%cd /content/LIMINAL

# OPTION B: Mount Google Drive if your LIMINAL directory is stored there
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/LIMINAL

# Verify current directory
!pwd
!ls -la

In [ ]:
# Install package in editable mode (safe to re-run; idempotent)
!pip install -e '.[dev]' -q

In [ ]:
# Confirm hardware
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU — training will be slow. Use Colab GPU runtime for Sections 2, 4, 7.')

---
## Section 1 — Phase 8 Housekeeping: P0 Metric Reconciliation

**Purpose**: Resolve the §18.4 discrepancy — `analyze_selectivity` reports 0.99 mean slots/step
for N=3 λ=0.001 while `analyze_gate_variance` implies 1.96. Both scripts run on the same checkpoint;
their outputs are compared inline.

No retraining needed. Uses existing checkpoint at `results/fair_hard_n3_lam001/best.pt`.

In [ ]:
# P0-A: Run analyze_selectivity (reports soft gate mean)
!python scripts/analyze_selectivity.py \
    --config  configs/experiments/fair_hard_n3_lam001.yaml \
    --checkpoint results/fair_hard_n3_lam001/best.pt \
    --output-dir results/phase7_eval/fair_hard_n3_lam001

In [ ]:
# P0-B: Run analyze_gate_variance (reports per-turn gate fractions)
!python scripts/analyze_gate_variance.py \
    --config  configs/experiments/fair_hard_n3_lam001.yaml \
    --checkpoint results/fair_hard_n3_lam001/best.pt \
    --output-dir results/variance/fair_hard_n3_lam001

In [ ]:
# P0-C: Load both JSON outputs and print comparison inline
import json
from pathlib import Path

sel_path  = Path('results/phase7_eval/fair_hard_n3_lam001')
var_path  = Path('results/variance/fair_hard_n3_lam001')

sel_files = list(sel_path.glob('*.json'))
var_files = list(var_path.glob('*.json'))

print('=== analyze_selectivity output ===')
if sel_files:
    with open(sel_files[0]) as f:
        sel = json.load(f)
    print(f"  mean_slots_written_per_step : {sel.get('mean_slots_written_per_step', 'N/A')}")
    print(f"  write_gate_mean (soft)      : {sel.get('write_gate_mean', 'N/A')}")
    print(f"  gate_entropy                : {sel.get('gate_entropy', 'N/A')}")
else:
    print('  No JSON found — re-run cell above')

print()
print('=== analyze_gate_variance output (per-turn means → implied slots) ===')
if var_files:
    with open(var_files[0]) as f:
        var = json.load(f)
    per_turn = var.get('per_turn', {})
    N = 3  # latent slots for this config
    fracs = [v.get('mean', 0) for v in per_turn.values()]
    if fracs:
        avg_frac = sum(fracs) / len(fracs)
        implied_slots = avg_frac * N
        print(f"  Per-turn mean fracs         : {[round(f,3) for f in fracs]}")
        print(f"  Average fraction            : {avg_frac:.4f}")
        print(f"  Implied mean slots (×N={N})  : {implied_slots:.4f}")
    else:
        print('  per_turn key missing from JSON')
else:
    print('  No JSON found — re-run cell above')

print()
print('CONCLUSION: If analyze_selectivity reports ~0.99 and implied slots above is ~1.96,')
print('the discrepancy is a denominator difference (soft gate mean vs hard gate × N).')

---
## Section 2 — Phase 8 Housekeeping: P3 Seed-43 P8-E Replications

**Purpose**: Condition 3 (cost-matched comparison) is ⚠️ because the N=8 P8-E run
converged at epoch 15 (suspect outlier). These seed-43 replications provide a second
data point at equal per-slot cost.

**Blocking**: Condition 3 stays provisional until both cells complete.  
**GPU time**: ~30 min per run on Colab T4.

In [ ]:
# Generate seed-43 YAML configs for both P8-E cells
import yaml, os
from pathlib import Path

def make_p8e_seed43_config(base_config_path: str, output_path: str, seed: int = 43):
    with open(base_config_path) as f:
        cfg = yaml.safe_load(f)
    cfg['seed'] = seed
    if 'data' in cfg:
        cfg['data']['seed'] = seed
    # Update checkpoint dir
    old_dir = cfg.get('training', {}).get('checkpoint_dir', '')
    cfg['training']['checkpoint_dir'] = old_dir.replace('p8e_matched', 'p8e_matched_seed43')
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f'Created: {output_path}')

make_p8e_seed43_config(
    'configs/experiments/p8e_matched_n3.yaml',
    'configs/experiments/p8e_matched_n3_seed43.yaml'
)
make_p8e_seed43_config(
    'configs/experiments/p8e_matched_n8.yaml',
    'configs/experiments/p8e_matched_n8_seed43.yaml'
)

In [ ]:
# P3-A: Train N=3, seed 43, matched per-slot cost
!python scripts/train_sequential.py \
    --config configs/experiments/p8e_matched_n3_seed43.yaml

In [ ]:
# P3-B: Train N=8, seed 43, matched per-slot cost
!python scripts/train_sequential.py \
    --config configs/experiments/p8e_matched_n8_seed43.yaml

In [ ]:
# P3-C: Selectivity analysis on N=3 seed-43 (includes write rates)
!python scripts/analyze_selectivity.py \
    --config  configs/experiments/p8e_matched_n3_seed43.yaml \
    --checkpoint results/p8e_matched_n3_seed43/best.pt \
    --output-dir results/phase7_eval/p8e_matched_n3_seed43

In [ ]:
# P3-D: Selectivity analysis on N=8 seed-43
!python scripts/analyze_selectivity.py \
    --config  configs/experiments/p8e_matched_n8_seed43.yaml \
    --checkpoint results/p8e_matched_n8_seed43/best.pt \
    --output-dir results/phase7_eval/p8e_matched_n8_seed43

In [ ]:
# P3-E: Compare seed-42 vs seed-43 P8-E results inline
import json
from pathlib import Path

runs = {
    'N3 seed-42 (original)': 'results/phase7_eval/p8e_matched_n3',
    'N3 seed-43 (new)':      'results/phase7_eval/p8e_matched_n3_seed43',
    'N8 seed-42 (original)': 'results/phase7_eval/p8e_matched_n8',  # may not exist
    'N8 seed-43 (new)':      'results/phase7_eval/p8e_matched_n8_seed43',
}

print(f'{"Run":<30} {"Test Acc":>10} {"Mean Slots":>12} {"Entropy":>10}')
print('-' * 65)
for label, path in runs.items():
    files = list(Path(path).glob('*.json')) if Path(path).exists() else []
    if files:
        with open(files[0]) as f:
            d = json.load(f)
        acc   = d.get('test_accuracy', d.get('learned_accuracy', 'N/A'))
        slots = d.get('mean_slots_written_per_step', 'N/A')
        entr  = d.get('gate_entropy', 'N/A')
        print(f'{label:<30} {str(acc):>10} {str(slots):>12} {str(entr):>10}')
    else:
        print(f'{label:<30} {"(not run yet)":>10}')

---
## Section 3 — Task Generator Validation

Validates both new Phase 9 task generators before any training.
Checks: answer correctness, label balance, turn structure, fact encoding.
**No GPU needed.**

In [ ]:
# Validate relational_graph generator across all difficulty levels
import sys
sys.path.insert(0, '.')
from src.data.relational_graph import RelationalGraphGenerator

for difficulty in ['easy', 'medium', 'hard', 'ood']:
    gen = RelationalGraphGenerator(difficulty=difficulty, seed=42,
                                   num_train=500, num_val=100, num_test=100)
    dataset = gen.generate_dataset()
    stats = gen.verify_dataset(dataset['test'])
    balance = stats['label_balance']
    pct_pos = balance.get(1, 0) / stats['n'] * 100
    print(f"[{difficulty:6s}] n={stats['n']}  "
          f"turns={stats['mean_turns']:.1f}  "
          f"edges={stats['mean_final_edges']:.1f}  "
          f"positive_labels={pct_pos:.1f}%")

print()
print('Sample sequence (medium, first test example):')
gen_m = RelationalGraphGenerator(difficulty='medium', seed=42, num_train=1, num_val=1, num_test=1)
sample = gen_m.generate_dataset()['test'][0]
print(f"  query_type : {sample['query_type']}")
print(f"  src→dst    : {sample['src_entity']} → {sample['dst_entity']}")
print(f"  path_length: {sample['path_length']}")
print(f"  num_turns  : {len(sample['turns'])}")
print(f"  ground_truth: {sample['turns'][-1]['ground_truth']}")
print(f"  proof_trace : {sample['turns'][-1]['proof_trace']}")

In [ ]:
# Validate obligation_network generator
from src.data.obligation_network import ObligationNetworkGenerator

for difficulty in ['medium', 'hard']:
    gen = ObligationNetworkGenerator(difficulty=difficulty, seed=42,
                                     num_train=500, num_val=100, num_test=100)
    dataset = gen.generate_dataset()
    stats = gen.verify_dataset(dataset['test'])
    balance = stats['label_balance']
    pct_pos = balance.get(1, 0) / stats['n'] * 100
    print(f"[{difficulty:6s}] n={stats['n']}  "
          f"turns={stats['mean_turns']:.1f}  "
          f"query_entities={stats['mean_query_entities']:.1f}  "
          f"positive_labels={pct_pos:.1f}%")

print()
print('Sample sequence (hard, first test example):')
gen_h = ObligationNetworkGenerator(difficulty='hard', seed=42, num_train=1, num_val=1, num_test=1)
sample = gen_h.generate_dataset()['test'][0]
print(f"  query_type       : {sample['query_type']}")
print(f"  num_query_entities: {sample['num_query_entities']}")
print(f"  num_distractors  : {sample['num_distractors']}")
print(f"  num_turns        : {len(sample['turns'])}")
print(f"  verified_gt      : {sample['verified_ground_truth']}")
print(f"  proof_trace      : {sample['turns'][-1]['proof_trace']}")

---
## Section 4 — Hypothesis A Training: `relational_graph`

Trains three models on the new relational graph task at Medium difficulty.
External workspace is **OFF** for all three runs — this section tests Hypothesis A only.

| Model | Config to create | Estimated GPU time |
|-------|-----------------|-------------------|
| A: Vector baseline | `rg_vector_medium.yaml` | ~20 min |
| B: Static graph | `rg_static_medium.yaml` | ~25 min |
| C: Persistent graph | `rg_persist_medium.yaml` | ~30 min |

**Note**: These configs need to be created for the new `relational_graph` task family.
The cell below auto-generates them from the existing base configs.

In [ ]:
# Auto-generate configs for Hypothesis A (relational_graph, medium difficulty)
import yaml
from pathlib import Path

BASE_DATA = {
    'task_family': 'relational_graph_medium',   # dispatched via ArithmeticGenerator
    'sequence_turns': 7,
    'ops_per_turn': 1,
    'incremental_turns': False,
    'num_train': 3000,
    'num_val': 500,
    'num_test': 500,
    'max_entities': 5,
    'max_operations': 2,
    'distractor_ratio': 0.0,
    'seed': 42,
}

BASE_TRAINING = {
    'epochs': 60,
    'batch_size': 8,
    'lr': 3.0e-4,
    'weight_decay': 1.0e-5,
    'grad_clip': 1.0,
    'scheduler': 'cosine',
    'log_every': 10,
    'eval_every': 1,
}

CONFIGS = [
    {
        'name': 'rg_vector_medium',
        'comment': 'Hypothesis A — Vector Baseline (Model A) on relational_graph medium',
        'model': {'type': 'vector', 'latent_slots': 1, 'latent_dim': 64,
                  'reasoning_steps': 4, 'max_reasoning_steps': 16,
                  'decoder_hidden_dim': 64, 'num_classes': 2,
                  'edge_dim': 1, 'msg_hidden_dim': 64, 'update_hidden_dim': 64, 'num_msg_layers': 1},
        'activity': {'enabled': False, 'gate_hidden_dim': 32, 'sparsity_lambda': 0.0, 'entropy_lambda': 0.0},
        'resolution': {'enabled': False, 'halt_threshold': 1.0, 'ponder_lambda': 0.0, 'max_reasoning_steps': 4},
        'external': {'enabled': False},
        'persistence': {'enabled': False},
        'checkpoint_dir': 'results/rg_vector_medium',
    },
    {
        'name': 'rg_static_medium',
        'comment': 'Hypothesis A — Static Graph (Model B) on relational_graph medium',
        'model': {'type': 'graph', 'latent_slots': 8, 'latent_dim': 32,
                  'edge_dim': 1, 'msg_hidden_dim': 64, 'update_hidden_dim': 64,
                  'num_msg_layers': 1, 'reasoning_steps': 4, 'max_reasoning_steps': 16,
                  'decoder_hidden_dim': 64, 'num_classes': 2},
        'activity': {'enabled': False, 'gate_hidden_dim': 32, 'sparsity_lambda': 0.0, 'entropy_lambda': 0.0},
        'resolution': {'enabled': False, 'halt_threshold': 1.0, 'ponder_lambda': 0.0, 'max_reasoning_steps': 4},
        'external': {'enabled': False},
        'persistence': {'enabled': False},
        'checkpoint_dir': 'results/rg_static_medium',
    },
    {
        'name': 'rg_persist_medium',
        'comment': 'Hypothesis A — Persistent Graph (Model C) on relational_graph medium',
        'model': {'type': 'graph', 'latent_slots': 8, 'latent_dim': 32,
                  'edge_dim': 1, 'msg_hidden_dim': 64, 'update_hidden_dim': 64,
                  'num_msg_layers': 1, 'reasoning_steps': 4, 'max_reasoning_steps': 16,
                  'decoder_hidden_dim': 64, 'num_classes': 2},
        'activity': {'enabled': True, 'gate_hidden_dim': 32, 'sparsity_lambda': 0.005, 'entropy_lambda': 0.001},
        'resolution': {'enabled': True, 'halt_threshold': 0.8, 'ponder_lambda': 0.01, 'max_reasoning_steps': 16},
        'external': {'enabled': False},
        'persistence': {'enabled': True, 'gate_hidden_dim': 32, 'detach_between_turns': True},
        'checkpoint_dir': 'results/rg_persist_medium',
    },
]

Path('configs/experiments').mkdir(parents=True, exist_ok=True)
for cfg in CONFIGS:
    out = {
        'model': cfg['model'],
        'activity': cfg['activity'],
        'resolution': cfg['resolution'],
        'external': cfg['external'],
        'persistence': cfg['persistence'],
        'data': BASE_DATA,
        'training': {**BASE_TRAINING, 'checkpoint_dir': cfg['checkpoint_dir']},
        'seed': 42,
    }
    path = f"configs/experiments/{cfg['name']}.yaml"
    with open(path, 'w') as f:
        f.write(f"# {cfg['comment']}\n")
        yaml.dump(out, f, default_flow_style=False)
    print(f'Created: {path}')

In [ ]:
# Train Model A: Vector Baseline
!python scripts/train_sequential.py \
    --config configs/experiments/rg_vector_medium.yaml

In [ ]:
# Train Model B: Static Graph
!python scripts/train_sequential.py \
    --config configs/experiments/rg_static_medium.yaml

In [ ]:
# Train Model C: Persistent Graph (primary Hypothesis A model)
!python scripts/train_sequential.py \
    --config configs/experiments/rg_persist_medium.yaml

---
## Section 5 — Hypothesis A Evaluation: Difficulty Ladder + Inference Scaling

Evaluates all three models on Easy/Medium/Hard/OOD difficulty splits.
Also sweeps `reasoning_steps` k ∈ {1,2,4,6,8} on Model C to test inference compute scaling.

In [ ]:
# Difficulty ladder evaluation — all models, all difficulty levels
import subprocess, json
from pathlib import Path

models = [
    ('Model A: Vector',    'configs/experiments/rg_vector_medium.yaml',  'results/rg_vector_medium/best.pt'),
    ('Model B: Static',    'configs/experiments/rg_static_medium.yaml',  'results/rg_static_medium/best.pt'),
    ('Model C: Persistent','configs/experiments/rg_persist_medium.yaml', 'results/rg_persist_medium/best.pt'),
]

difficulties = ['easy', 'medium', 'hard', 'ood']
results = {}

for model_name, config, ckpt in models:
    results[model_name] = {}
    for diff in difficulties:
        out_dir = f'results/rg_ladder/{model_name.split(":")[0].strip().lower().replace(" ","_")}/{diff}'
        # Use analyze_selectivity as the eval harness (works for any sequential model)
        cmd = [
            'python', 'scripts/analyze_selectivity.py',
            '--config', config,
            '--checkpoint', ckpt,
            '--output-dir', out_dir,
        ]
        print(f'  Evaluating {model_name} @ {diff} ...')
        proc = subprocess.run(cmd, capture_output=True, text=True)
        # Parse accuracy from output
        for line in proc.stdout.splitlines():
            if 'accuracy' in line.lower() or 'test acc' in line.lower():
                print(f'    {line.strip()}')
        json_files = list(Path(out_dir).glob('*.json')) if Path(out_dir).exists() else []
        if json_files:
            with open(json_files[0]) as f:
                d = json.load(f)
            results[model_name][diff] = d.get('test_accuracy', d.get('learned_accuracy', None))
        else:
            results[model_name][diff] = None

# Print summary table
print()
print(f'{"Model":<25}', end='')
for d in difficulties:
    print(f'{d:>10}', end='')
print()
print('-' * (25 + 10 * len(difficulties)))
for model_name, accs in results.items():
    print(f'{model_name:<25}', end='')
    for d in difficulties:
        v = accs.get(d)
        print(f'{(str(round(v*100,1))+"%" if v else "N/A"):>10}', end='')
    print()

In [ ]:
# [OPTIONAL] Inference compute scaling — vary reasoning_steps k on Model C
# Tests whether more latent computation improves accuracy on Hard/OOD splits
import subprocess, json
from pathlib import Path

k_values = [1, 2, 4, 6, 8]
scaling_results = {}

for k in k_values:
    out_dir = f'results/rg_inference_scaling/k{k}'
    # Pass reasoning_steps override via a temp config
    import yaml
    with open('configs/experiments/rg_persist_medium.yaml') as f:
        cfg = yaml.safe_load(f)
    cfg['model']['reasoning_steps'] = k
    tmp_path = f'/tmp/rg_persist_k{k}.yaml'
    with open(tmp_path, 'w') as f:
        yaml.dump(cfg, f)

    cmd = [
        'python', 'scripts/analyze_selectivity.py',
        '--config', tmp_path,
        '--checkpoint', 'results/rg_persist_medium/best.pt',
        '--output-dir', out_dir,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    json_files = list(Path(out_dir).glob('*.json')) if Path(out_dir).exists() else []
    if json_files:
        with open(json_files[0]) as f:
            d = json.load(f)
        acc = d.get('test_accuracy', d.get('learned_accuracy', None))
        scaling_results[k] = acc
        print(f'  k={k}: acc={round(acc*100,2) if acc else "N/A"}%')

# Plot
try:
    import matplotlib.pyplot as plt
    ks = [k for k in k_values if scaling_results.get(k)]
    accs = [scaling_results[k]*100 for k in ks]
    plt.figure(figsize=(6,4))
    plt.plot(ks, accs, 'o-', color='steelblue', linewidth=2)
    plt.xlabel('Reasoning steps (k)')
    plt.ylabel('Test accuracy (%)')
    plt.title('Inference Compute Scaling — Model C (Persistent Graph)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/rg_inference_scaling/scaling_curve.png', dpi=150)
    plt.show()
    print('Plot saved to results/rg_inference_scaling/scaling_curve.png')
except Exception as e:
    print(f'Plot failed: {e}')

---
## Section 6 — Hypothesis A Evaluation: Slot Specialisation Probe `[OPTIONAL]`

Trains a linear probe on frozen Model C slot states to test whether different slots
preferentially represent different entities. Addresses the open question in `hypothesis.md`:
*"we did not pretend to know what a latent node represented."*

In [ ]:
# Slot specialisation probe — linear probe: slot_i state → entity_j identity
import torch, torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from src.utils.config import load_config, set_seed
from src.data.relational_graph import RelationalGraphGenerator
from src.data.dataset import Vocabulary
from src.data.sequence_dataset import SequenceReasoningDataset, collate_sequence_batch
from src.models.sequential_model import SequentialReasoningModel
from src.utils.checkpoint import load_checkpoint

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
import yaml
from pathlib import Path
with open('configs/experiments/rg_persist_medium.yaml') as f:
    raw_cfg = yaml.safe_load(f)
from src.utils.config import load_config
config = load_config('configs/base.yaml', overrides=raw_cfg)

model = SequentialReasoningModel(config).to(device)
ckpt_path = 'results/rg_persist_medium/best.pt'
if Path(ckpt_path).exists():
    load_checkpoint(ckpt_path, model)
    model.eval()
    print('Model loaded successfully')

    # Generate test data and collect slot states
    gen = RelationalGraphGenerator(difficulty='medium', seed=42, num_train=1, num_val=1, num_test=200)
    test_data = gen.generate_dataset()['test']
    vocab = Vocabulary()
    dataset = SequenceReasoningDataset(test_data, vocab)
    loader = DataLoader(dataset, batch_size=8, collate_fn=collate_sequence_batch)

    # Collect (slot_state, entity_label) pairs
    slot_states = []  # [N_examples * N_turns, N_slots, latent_dim]
    entity_labels = []  # [N_examples * N_turns, N_slots]  — which entity a slot was loaded with

    hooks = []
    captured = []

    def hook_fn(module, input, output):
        # Capture latent slot states after message passing
        if isinstance(output, torch.Tensor) and output.dim() == 3:
            captured.append(output.detach().cpu())

    # Register hook on the latent workspace module if available
    for name, module in model.named_modules():
        if 'latent_workspace' in name or 'workspace' in name.lower():
            h = module.register_forward_hook(hook_fn)
            hooks.append(h)
            print(f'Hooked: {name}')
            break

    with torch.no_grad():
        for batch in loader:
            captured.clear()
            facts = batch['facts'].to(device)
            fact_mask = batch['fact_mask'].to(device)
            _ = model(facts, fact_mask)
            if captured:
                slot_states.append(captured[-1])  # last captured state

    for h in hooks:
        h.remove()

    if slot_states:
        all_states = torch.cat(slot_states, dim=0)  # [B_total, N_slots, latent_dim]
        N_slots = all_states.shape[1]
        latent_dim = all_states.shape[2]
        print(f'Collected {all_states.shape[0]} slot state tensors')
        print(f'Slots: {N_slots}, Latent dim: {latent_dim}')

        # Train a simple linear probe per slot
        # (probe predicts which entity was most recently written to each slot)
        # Since ground-truth entity→slot assignment isn't stored, we use
        # cluster assignment via k-means as a proxy for specialisation
        from sklearn.cluster import KMeans
        from sklearn.metrics import silhouette_score

        print('\nSlot specialisation analysis (k-means silhouette per slot):')
        for s in range(N_slots):
            slot_data = all_states[:, s, :].numpy()  # [B_total, latent_dim]
            if slot_data.shape[0] > 10:
                km = KMeans(n_clusters=min(5, slot_data.shape[0]//2), random_state=42, n_init=5)
                labels = km.fit_predict(slot_data)
                sil = silhouette_score(slot_data, labels) if len(set(labels)) > 1 else 0.0
                print(f'  Slot {s}: silhouette = {sil:.4f}  ({"specialised" if sil > 0.3 else "diffuse"})')
    else:
        print('No slot states captured — check hook registration')
else:
    print(f'Checkpoint not found at {ckpt_path} — run Section 4 first')

---
## Section 7 — Hypothesis B Training: `obligation_network`

Trains two models on the new obligation network task (hard difficulty).
The internal-only model (Model C variant) is the baseline; Full LIMINAL (Model D) is the claim.

| Model | Config | Estimated GPU time |
|-------|--------|-------------------|
| C: Persistent (no external) | `on_persist_hard.yaml` | ~30 min |
| D: Full LIMINAL (learned gate) | `on_liminal_hard.yaml` | ~45 min |

In [ ]:
# Auto-generate configs for Hypothesis B (obligation_network, hard difficulty)
import yaml
from pathlib import Path

BASE_DATA_B = {
    'task_family': 'obligation_network_hard',
    'sequence_turns': 12,
    'ops_per_turn': 2,
    'incremental_turns': True,
    'num_train': 3000,
    'num_val': 500,
    'num_test': 500,
    'max_entities': 8,
    'max_operations': 2,
    'distractor_ratio': 0.0,
    'seed': 42,
}

BASE_MODEL = {
    'type': 'graph', 'latent_slots': 3, 'latent_dim': 32,
    'edge_dim': 1, 'msg_hidden_dim': 64, 'update_hidden_dim': 64,
    'num_msg_layers': 1, 'reasoning_steps': 4, 'max_reasoning_steps': 16,
    'decoder_hidden_dim': 64, 'num_classes': 2,
}

# Model C: Persistent graph, no external workspace
cfg_c = {
    'model': BASE_MODEL,
    'activity': {'enabled': True, 'gate_hidden_dim': 32, 'sparsity_lambda': 0.005, 'entropy_lambda': 0.001},
    'resolution': {'enabled': True, 'halt_threshold': 0.8, 'ponder_lambda': 0.01, 'max_reasoning_steps': 16},
    'external': {'enabled': False},
    'persistence': {'enabled': True, 'gate_hidden_dim': 32, 'detach_between_turns': True},
    'data': BASE_DATA_B,
    'training': {'epochs': 60, 'batch_size': 8, 'lr': 3.0e-4, 'weight_decay': 1.0e-5,
                 'grad_clip': 1.0, 'scheduler': 'cosine', 'log_every': 10, 'eval_every': 1,
                 'checkpoint_dir': 'results/on_persist_hard'},
    'seed': 42,
}

# Model D: Full LIMINAL with learned write gate
cfg_d = {
    'model': BASE_MODEL,
    'activity': {'enabled': True, 'gate_hidden_dim': 32, 'sparsity_lambda': 0.005, 'entropy_lambda': 0.001},
    'resolution': {'enabled': True, 'halt_threshold': 0.8, 'ponder_lambda': 0.01, 'max_reasoning_steps': 16},
    'external': {
        'enabled': True,
        'num_slots': 16, 'record_dim': 32, 'num_types': 7, 'write_top_k': 3,
        'write_dedup': True, 'read_heads': 2, 'read_gate': True,
        'edge_adaptation': True, 'edge_hidden_dim': 32,
        'detach_workspace_between_turns': True,
        'learned_gate': True,
        'gumbel_tau_start': 1.0, 'gumbel_tau_end': 0.1, 'gumbel_anneal_epochs': 30,
        'write_sparsity_lambda': 0.001,
        'write_gate_bias_init': 0.0,
        'write_sparsity_warmup_epochs': 5,
    },
    'persistence': {'enabled': True, 'gate_hidden_dim': 32, 'detach_between_turns': True},
    'data': BASE_DATA_B,
    'training': {'epochs': 60, 'batch_size': 8, 'lr': 3.0e-4, 'weight_decay': 1.0e-5,
                 'grad_clip': 1.0, 'scheduler': 'cosine', 'log_every': 10, 'eval_every': 1,
                 'checkpoint_dir': 'results/on_liminal_hard'},
    'seed': 42,
}

for cfg, fname, label in [
    (cfg_c, 'on_persist_hard', 'Model C — Persistent (no external)'),
    (cfg_d, 'on_liminal_hard', 'Model D — Full LIMINAL (learned gate)'),
]:
    path = f'configs/experiments/{fname}.yaml'
    with open(path, 'w') as f:
        f.write(f'# Hypothesis B — {label} on obligation_network hard\n')
        yaml.dump(cfg, f, default_flow_style=False)
    print(f'Created: {path}')

In [ ]:
# Train Model C: Persistent (no external) — Hypothesis B internal-only baseline
!python scripts/train_sequential.py \
    --config configs/experiments/on_persist_hard.yaml

In [ ]:
# Train Model D: Full LIMINAL with learned write gate
!python scripts/train_sequential.py \
    --config configs/experiments/on_liminal_hard.yaml

---
## Section 8 — Hypothesis B Evaluation: 4-Condition Write-Policy Comparison

Evaluates Model D under four write conditions on the same frozen checkpoint:
1. **NORMAL** — learned gate as trained
2. **ZERO_WRITE** — gate forced to −∞ (no external writes)
3. **CORRUPT** — gate fires normally; records zeroed before reads
4. **TOP_K** — forced K=3 writes per step (heuristic baseline)

In [ ]:
# Selectivity analysis: gate stats for Model D (learned gate)
!python scripts/analyze_selectivity.py \
    --config  configs/experiments/on_liminal_hard.yaml \
    --checkpoint results/on_liminal_hard/best.pt \
    --output-dir results/on_selectivity/liminal_hard

In [ ]:
# Zero-write ablation: NORMAL vs ZERO_WRITE vs CORRUPT
!python scripts/ablate_zero_write.py \
    --config  configs/experiments/on_liminal_hard.yaml \
    --checkpoint results/on_liminal_hard/best.pt \
    --output-dir results/on_ablations/liminal_hard

In [ ]:
# Print 4-condition comparison inline
import json
from pathlib import Path

# Load selectivity (NORMAL + TOP_K)
sel_files = list(Path('results/on_selectivity/liminal_hard').glob('*.json'))
abl_files = list(Path('results/on_ablations/liminal_hard').glob('*.json'))

print('=== 4-Condition Write-Policy Comparison — Model D on obligation_network/hard ===')
print()

if sel_files:
    with open(sel_files[0]) as f:
        sel = json.load(f)
    print(f"NORMAL   (learned gate) : {sel.get('learned_accuracy', sel.get('test_accuracy', 'N/A')):.4f}")
    print(f"TOP_K    (forced k=3)   : {sel.get('baseline_accuracy', 'N/A')}")
    print(f"Mean slots/step         : {sel.get('mean_slots_written_per_step', 'N/A')}")
    print(f"Gate entropy            : {sel.get('gate_entropy', 'N/A')}")
    print(f"Gate status             : {sel.get('gate_status', 'N/A')}")

print()
if abl_files:
    with open(abl_files[0]) as f:
        abl = json.load(f)
    for condition in ['NORMAL', 'ZERO_WRITE', 'CORRUPT']:
        acc = abl.get(condition, {}).get('overall_accuracy', 'N/A')
        print(f"{condition:<12} : {acc}")

# Load Model C (internal-only) for comparison
c_files = list(Path('results/on_selectivity/persist_hard').glob('*.json')) if Path('results/on_selectivity/persist_hard').exists() else []
if c_files:
    with open(c_files[0]) as f:
        c_data = json.load(f)
    print(f"\nModel C (no external)   : {c_data.get('test_accuracy', 'N/A')}")

---
## Section 9 — Hypothesis B Evaluation: Demand-Sensitivity + Per-Turn Ablation

Tests whether the gate writes more on examples where writing changes the answer
(three-way demand analysis), and maps the per-turn causal chain of writes.

In [ ]:
# Three-way demand sensitivity analysis (corrected design from §18.3)
!python scripts/analyze_three_way_demand.py \
    --config  configs/experiments/on_liminal_hard.yaml \
    --checkpoint results/on_liminal_hard/best.pt \
    --output-dir results/on_demand/liminal_hard

In [ ]:
# Per-turn workspace ablation (Condition 2 — causal chain)
!python scripts/analyze_per_turn_ablation.py \
    --config  configs/experiments/on_liminal_hard.yaml \
    --checkpoint results/on_liminal_hard/best.pt \
    --output-dir results/on_per_turn/liminal_hard

In [ ]:
# Gate variance analysis (content vs positional schedule)
!python scripts/analyze_gate_variance.py \
    --config  configs/experiments/on_liminal_hard.yaml \
    --checkpoint results/on_liminal_hard/best.pt \
    --output-dir results/on_variance/liminal_hard

In [ ]:
# Print demand-sensitivity summary inline
import json
from pathlib import Path

demand_dir = Path('results/on_demand/liminal_hard')
demand_files = list(demand_dir.glob('*.json')) if demand_dir.exists() else []

if demand_files:
    with open(demand_files[0]) as f:
        d = json.load(f)

    groups = d.get('group_sizes', {})
    print(f"Group composition:")
    print(f"  helped       : {groups.get('helped', 'N/A')}")
    print(f"  both-correct : {groups.get('both_correct', 'N/A')}")
    print(f"  normal-wrong : {groups.get('normal_wrong', 'N/A')}")
    print()
    print(f"Demand verdict: {d.get('verdict', 'N/A')}")
    print(f"Significant positive turns: {d.get('significant_positive_turns', 'N/A')}")

    per_turn = d.get('per_turn', {})
    print()
    print(f"{'Turn':<6} {'N_hlp':>7} {'N_ok':>7} {'hlp':>8} {'ok':>8} {'delta':>8} {'sig':>5}")
    print('-' * 52)
    for t, stats in per_turn.items():
        sig = stats.get('significant', False)
        direction = stats.get('direction', '')
        sig_str = ('*' + direction[:3]) if sig else ''
        print(f"{t:<6} {stats.get('n_helped',''):>7} {stats.get('n_both_correct',''):>7} "
              f"{stats.get('mean_helped', 0):>8.4f} {stats.get('mean_both_correct', 0):>8.4f} "
              f"{stats.get('delta', 0):>+8.4f} {sig_str:>5}")
else:
    print('No demand analysis JSON found — run cell above first')

---
## Section 10 — Final Scorecard

Aggregates results from all sections into the Phase 9 LM-transition scorecard.

In [ ]:
# Final scorecard — reads existing JSON outputs and populates the table
import json
from pathlib import Path

def load_first_json(directory):
    d = Path(directory)
    if d.exists():
        files = list(d.glob('*.json'))
        if files:
            with open(files[0]) as f:
                return json.load(f)
    return {}

print('=' * 70)
print('LIMINAL Phase 9 — LM Transition Scorecard')
print('=' * 70)

# ── Hypothesis A ────────────────────────────────────────────────────────
print('\n[HYPOTHESIS A: Dynamic Latent Reasoning]')
print(f"  Task: relational_graph (Medium difficulty)")

for label, path in [
    ('Model A (Vector)',    'results/rg_ladder/model_a/medium'),
    ('Model B (Static)',    'results/rg_ladder/model_b/medium'),
    ('Model C (Persist)',   'results/rg_ladder/model_c/medium'),
]:
    d = load_first_json(path)
    acc = d.get('test_accuracy', d.get('learned_accuracy', 'not run'))
    print(f"  {label:<25}: {acc}")

# ── Hypothesis B ────────────────────────────────────────────────────────
print('\n[HYPOTHESIS B: Internal–External Co-refinement]')
print(f"  Task: obligation_network (Hard difficulty)")

for label, path in [
    ('Model C (no external)', 'results/on_selectivity/persist_hard'),
    ('Model D NORMAL',        'results/on_selectivity/liminal_hard'),
]:
    d = load_first_json(path)
    acc = d.get('test_accuracy', d.get('learned_accuracy', 'not run'))
    print(f"  {label:<25}: {acc}")

abl = load_first_json('results/on_ablations/liminal_hard')
for cond in ['ZERO_WRITE', 'CORRUPT']:
    acc = abl.get(cond, {}).get('overall_accuracy', 'not run')
    print(f"  {'Model D ' + cond:<25}: {acc}")

dem = load_first_json('results/on_demand/liminal_hard')
print(f"\n  Demand verdict   : {dem.get('verdict', 'not run')}")
print(f"  Sig positive turns: {dem.get('significant_positive_turns', 'not run')} / {dem.get('total_turns', 'N/A')}")

# ── P8 Housekeeping ─────────────────────────────────────────────────────
print('\n[PHASE 8 HOUSEKEEPING]')
p0_sel = load_first_json('results/phase7_eval/fair_hard_n3_lam001')
print(f"  P0 soft mean slots (analyze_selectivity): {p0_sel.get('mean_slots_written_per_step', 'not run')}")
p3_n3 = load_first_json('results/phase7_eval/p8e_matched_n3_seed43')
p3_n8 = load_first_json('results/phase7_eval/p8e_matched_n8_seed43')
print(f"  P3 N=3 seed-43 acc: {p3_n3.get('test_accuracy', p3_n3.get('learned_accuracy', 'not run'))}")
print(f"  P3 N=8 seed-43 acc: {p3_n8.get('test_accuracy', p3_n8.get('learned_accuracy', 'not run'))}")

print()
print('Run all sections above then re-run this cell for a complete scorecard.')
print('=' * 70)